In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import pandas as pd

#Download data from yfinance
#Take Apple as our targest, simulate its future
appl = yf.download('AAPL', start = '2021-08-05', end ='2026-08-06', auto_adjust = False, multi_level_index=False)
appl_adj = appl['Adj Close']

print(type(appl_adj))
#Calc mu- mean_daily_return and sigma-mean_variance
daily_returns = appl_adj.pct_change().dropna()

daily_mu = daily_returns.mean()

daily_sigma = daily_returns.std(ddof=1)

#annualised_mu= daily_mu*252
ann_mu = daily_mu*252
ann_sigma = daily_sigma*np.sqrt(252)

S0 = appl_adj.iloc[-1]


#Piece 1: 
def simulate_gbm(S0, ann_mu, ann_sigma, days = 252, n_paths=1000, dt=1/252):
    #Piece 2: generating random shock
    e = np.random.normal(0,1,(days,n_paths))
    log_steps = (ann_mu - (ann_sigma**2)/2)*dt + ann_sigma*e*(np.sqrt(dt))
    log_paths = np.cumsum(log_steps, axis=0)
    paths = np.exp(log_paths)*S0
    paths = np.vstack([np.full(n_paths,S0),paths])
    price_returns= paths[-1,:]
    # Alternatively: paths = np.insert(paths, 0, [S0]*1000, axis = 0)
    pct_returns = (paths[-1,:]/paths[0,:]-1)*100
    print('mean:', pct_returns.mean())
    print('std:', pct_returns.std())
    return price_returns


price_returns = simulate_gbm(S0, ann_mu, ann_sigma, days = 252, n_paths = 1000, dt =1/252)
sorted_paths=np.sort(price_returns) #np.sort(paths)[::-1] for descending order
median = (sorted_paths[int(len(sorted_paths)/2)]+sorted_paths[int(len(sorted_paths)/2)+1])/2
print(median)
distances = paths - median
possible_idx = distances.argmin()
print(paths[possible_idx])


[*********************100%***********************]  1 of 1 completed


<class 'pandas.Series'>
mean: 19.208852379802007
std: 33.55187229888093
357.0313840142475
